In [2]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing


housing=fetch_california_housing()
print(housing)


{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]], shape=(20640, 8)), 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)), 'frame': None, 'target_names': ['MedHouseVal'], 'feature_names': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'], 'DESCR': '.. _california_housing_dataset

In [3]:
housing

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]], shape=(20640, 8)),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': 

In [4]:
# Preparing the dataset

data=pd.DataFrame(housing.data,columns=housing.feature_names)
data.sample(10)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
10037,1.4241,16.0,4.285285,0.957958,642.0,1.927928,39.23,-121.02
18603,7.1366,23.0,5.685393,1.078652,264.0,2.966292,37.15,-122.08
19725,1.9797,36.0,5.742138,1.034591,1068.0,3.358491,38.96,-121.62
15802,4.3232,28.0,3.732021,1.045420,2232.0,1.689629,37.75,-122.44
13306,3.1250,22.0,4.714542,1.132855,1678.0,3.012567,34.08,-117.62
19410,4.4000,25.0,7.230216,1.143885,390.0,2.805755,37.70,-120.94
18053,10.7582,25.0,8.715000,1.060000,604.0,3.020000,37.25,-121.99
9368,5.0779,34.0,5.487179,0.990138,1219.0,2.404339,37.93,-122.52
14959,5.1221,4.0,6.176179,1.025227,6945.0,2.872208,32.75,-116.94
17909,4.5750,36.0,5.625000,0.985294,306.0,2.250000,37.36,-121.97


In [5]:
data['price']=housing.target
data.sample(10)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
14341,3.1979,43.0,3.791557,1.071240,692.0,1.825858,32.74,-117.16,4.667
3880,4.7000,36.0,6.020101,1.057789,1064.0,2.673367,34.21,-118.51,2.236
15567,5.4663,25.0,7.498807,1.064439,1286.0,3.069212,33.12,-117.03,2.483
16852,3.4300,46.0,5.438438,1.012012,796.0,2.390390,37.63,-122.42,2.929
16216,1.7885,36.0,5.151724,1.303448,338.0,2.331034,37.95,-121.32,0.621
15896,1.1125,28.0,5.548276,1.172414,1064.0,3.668966,37.72,-122.39,2.063
9326,3.5588,37.0,4.267516,1.025478,621.0,1.977707,37.97,-122.53,2.688
9745,3.3456,37.0,4.514085,0.908451,458.0,3.225352,36.67,-121.70,2.526
19253,4.7450,28.0,6.313933,1.077601,1634.0,2.881834,38.46,-122.81,2.486
14023,2.2687,34.0,3.642857,1.138776,804.0,1.640816,32.75,-117.16,2.000


In [6]:
from urllib.parse import urlparse
X=data.drop(columns=['price'])
y=data['price']


In [7]:
xtrain,xtest,ytrain,ytest=train_test_split(X,y,random_state=42,test_size=0.2)
from mlflow.models import infer_signature
signature=infer_signature(xtrain,ytrain)

param_grid={
    'n_estimators':[100,200],
    'max_depth':[5,10,None],
    'min_samples_split':[2,5],
    'min_samples_leaf':[1,2]
}

In [13]:
# Hyper parameter tuning using gridsearchCV

def hyperParameterTuner(xtrain,ytrain,param_grid):
    rf=RandomForestRegressor()
    gridsearch=GridSearchCV(estimator=rf,param_grid=param_grid,cv=5,n_jobs=1,verbose=2,scoring="neg_mean_squared_error")

    gridsearch.fit(xtrain,ytrain)
    return gridsearch

In [15]:
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")
with mlflow.start_run():
    grid_search=hyperParameterTuner(xtrain,ytrain,param_grid)
    # get the best model
    best_model=grid_search.best_estimator_
    # evaluate the best model
    ypred=best_model.predict(xtest)
    mse=mean_squared_error(ytest,ypred)
    # log best parameters and metrics
    mlflow.log_param("best_n_estimators",grid_search.best_params_['n_estimators'])
    mlflow.log_param("best_max_depth",grid_search.best_params_['max_depth'])
    mlflow.log_param("best_min_samples_split",grid_search.best_params_['min_samples_split'])
    mlflow.log_param("best_min_samples_leaf",grid_search.best_params_['min_samples_leaf'])
    mlflow.log_metric("mse",mse)

    # Tracking uri
    tracking_url_type_store=urlparse(mlflow.get_tracking_uri()).scheme

    if tracking_url_type_store != 'file':
        mlflow.sklearn.log_model(best_model,"model",registered_model_name="bestRandomForestModel")
    else:
        mlflow.sklearn.log_model(best_model,"model",signature=signature)

    print(f'Best HyperParameters:{grid_search.best_params_}')
    print(f'Mean squared error: {mse}')

Fitting 5 folds for each of 24 candidates, totalling 120 fits
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   3.0s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.8s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.8s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.9s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.8s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   5.6s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   5.5s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   5.9s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   5.6s
[CV] END max_depth=5, min_samples_leaf

2026/09/01 12:45:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'bestRandomForestModel'.
2026/09/01 12:45:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bestRandomForestModel, version 1
Created version '1' of model 'bestRandomForestModel'.


Best HyperParameters:{'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Mean squared error: 0.25288539690208595
🏃 View run enthused-frog-888 at: http://127.0.0.1:5000/#/experiments/0/runs/a67d711cb12c4c6b8c928a8784cc5fbd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
